In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.utils import class_weight

# Verify GPUs are recognized
gpus = tf.config.list_physical_devices('GPU')
print(f"✅ Success! Num GPUs Available: {len(gpus)}")
if len(gpus) > 0:
    print("Using Hardware Accelerator: GPU")

2026-05-21 13:49:09.513061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779371349.542262     323 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779371349.550396     323 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779371349.571005     323 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779371349.571029     323 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779371349.571031     323 computation_placer.cc:177] computation placer alr

✅ Success! Num GPUs Available: 2
Using Hardware Accelerator: GPU


In [2]:
# The absolute path based on your exact Kaggle UI folder structure
base_dir = '/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/augmented_resized_V2/train'

# Simple data augmentation optimized for speed on dual GPUs
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # Uses 20% of the train folder for validation
    horizontal_flip=True,
    vertical_flip=True
)

# Training Data Loader
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=64,         # Boosted to 64 since you have dual GPUs
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation Data Loader
val_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Verification check: This MUST print "Found X images belonging to 5 classes"
print("Class Mapping Detected:", train_generator.class_indices)

Found 92194 images belonging to 5 classes.
Found 23047 images belonging to 5 classes.
Class Mapping Detected: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}


In [3]:
# Extract classes array from generator
class_indices = train_generator.classes
unique_classes = np.unique(class_indices)

# Compute balanced weights
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=class_indices
)

class_weight_dict = dict(zip(unique_classes, weights))

print("✅ Calculated 5-Class Weight Structure:")
for cls, weight in class_weight_dict.items():
    print(f"  Class {cls}: {weight:.4f}")

✅ Calculated 5-Class Weight Structure:
  Class 0: 0.4178
  Class 1: 1.2479
  Class 2: 0.9525
  Class 3: 2.9042
  Class 4: 2.4326


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Initialize the Multi-GPU Distribution System
strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices in strategy: {strategy.num_replicas_in_sync}")

# 2. Build and compile the model INSIDE the strategy scope
with strategy.scope():
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(5, activation='softmax')
    ])

    model.compile(
        optimizer='adam', 
        loss='categorical_crossentropy', 
        metrics=['accuracy']
)

print("🚀 Starting accelerated Dual-GPU training...")

# 3. Execute training
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    class_weight=class_weight_dict
)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices in strategy: 2


I0000 00:00:1779371929.272661     323 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779371929.277853     323 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 Starting accelerated Dual-GPU training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1779371934.212099     398 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779371935.788714     400 cuda_dnn.cc:529] Loaded cuDNN version 91002


1441/1441 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step - accuracy: 0.3968 - loss: 1.5385INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 653s 448ms/step - accuracy: 0.3969 - loss: 1.5384 - val_accuracy: 0.4320 - val_loss: 1.4594
Epoch 2/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 651s 451ms/step - accuracy: 0.5226 - loss: 1.3819 - val_accuracy: 0.5917 - val_loss: 1.1112
Epoch 3/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 653s 453ms/step - accuracy: 0.5742 - loss: 1.2537 - val_accuracy: 0.6044 - val_loss: 1.0515
Epoch 4/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 677s 470ms/step - accuracy: 0.6088 - loss: 1.1450 - val_accuracy: 0.6012 - val_loss: 1.0808
Epoch 5/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 670s 465ms/step - accuracy: 0.6361 - loss: 1.0480 - val_accuracy: 0.6295 - val_loss: 1.0128
Epoch 6/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 657s 455ms/step - accuracy: 0.6592 - loss: 0.9662 - val_accuracy: 0.6298 - val_lo

In [5]:
# 1. Save the final weights explicitly
model.save('visionid_v1.h5')
print("✅ Model successfully saved as visionid_v1.h5")

# 2. Print out the exact mapping so you can copy it for your backend service
print("\n🔥 COPY THIS EXACT DICTIONARY FOR THE BACKEND:")
print(train_generator.class_indices)

✅ Model successfully saved as visionid_v1.h5

🔥 COPY THIS EXACT DICTIONARY FOR THE BACKEND:
{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}
